In [ ]:
# === Importaciones ===
import os, json, time, zipfile, textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    confusion_matrix, classification_report, balanced_accuracy_score
)


In [ ]:
Train = pd.read_csv("T_train_final_objetivo.csv")
Test  = pd.read_csv("T_test_final_objetivo.csv")

X_train = Train.iloc[:, :-1].copy()
y_train = Train.iloc[:, -1].astype(str)
X_test  = Test.iloc[:, :-1].copy()
y_test  = Test.iloc[:, -1].astype(str)

print("Formas:", X_train.shape, X_test.shape, "| Clases (train):", sorted(pd.unique(y_train)))


In [ ]:
CONFIG = {
    "usuario_declara_desbalance": None,   # None/True/False
    "importa_distinguir_clases": False,   # True si los costes por clase importan
    "top_k": None,                        # p.ej. 3 para Top-3 accuracy; None para no usar
    "rare_threshold": 0.05,               # clases raras si < 5%

    # Grid de hiperparámetros del árbol:
    "grid": {
        "criterion": ["gini", "entropy", "log_loss"],
        "max_depth": [None, 3, 5, 7, 9, 12],
        "min_samples_leaf": [1, 3, 5, 10],
        "class_weight": [None, "balanced"]
    },
    "cv_folds": 5,
    "random_state": 0,

    # Visualización del árbol truncado (para lectura humana)
    "max_depth_visual": 3,

    # Convención de prefijos dummy para agregación (opcional)
    "SEP": "___",

    # Carpeta de salida
    "OUTDIR": "dt_multiclase_artifacts"
}
OUTDIR = Path(CONFIG["OUTDIR"]); OUTDIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# === UTILIDADES ===
def diagnostico_balance_multiclase(y, rare_threshold=0.05):
    y_series = pd.Series(y)
    vc = y_series.value_counts(dropna=False).sort_index()
    n = int(vc.sum()); k = int(vc.shape[0])
    tabla = pd.DataFrame({"clase": vc.index, "n": vc.values, "pct": vc.values / n})
    n_min, n_max = tabla["n"].min(), tabla["n"].max()
    IR = (n_max / n_min) if n_min > 0 else np.inf
    if IR < 1.5:
        etiqueta = "Balance razonable (IR < 1.5)"
    elif IR < 3:
        etiqueta = "Desbalance moderado (1.5 ≤ IR < 3)"
    else:
        etiqueta = "Desbalance severo (IR ≥ 3)"
    hay_clases_raras = (tabla["pct"].min() < rare_threshold)
    recomendar_estratificar = (IR >= 1.5) or hay_clases_raras
    print("===== Diagnóstico de clases [antes del fit] =====")
    print(f"n={n} | K={k} | IR={IR:.3f} -> {etiqueta}")
    for _, row in tabla.iterrows():
        print(f"Clase {row['clase']}: n={int(row['n'])} ({row['pct']:.1%})")
    if hay_clases_raras:
        clases_raras = tabla.loc[tabla["pct"] < rare_threshold, "clase"].tolist()
        print(f"⚠︎ Clases raras (<{rare_threshold:.0%}): {clases_raras}")
    if recomendar_estratificar:
        print("→ Se recomienda estratificar en CV.")
    return {
        "tabla": tabla, "n": n, "K": k, "IR": IR,
        "etiqueta": etiqueta, "clases_raras": tabla.loc[tabla["pct"] < rare_threshold, "clase"].tolist(),
        "recomendar_estratificar": recomendar_estratificar
    }

def decidir_metricas(K, diag, config):
    if config["usuario_declara_desbalance"] is not None:
        desbalance = bool(config["usuario_declara_desbalance"])
        razon = "forzado_por_usuario"
    else:
        desbalance = (diag["IR"] >= 1.5) or (len(diag["clases_raras"]) > 0)
        razon = "diagnostico_automatico"

    print(f"\n>>> Decisión de balance: desbalance={desbalance} (razón={razon})")
    importa_costes = bool(config["importa_distinguir_clases"])
    print(f">>> Importa distinguir entre clases (costes distintos): {importa_costes}")

    if K == 2:
        scoring_cv = "f1" if (desbalance or importa_costes) else "accuracy"
        plan = {"modo": "binario", "desbalance": desbalance, "importa_costes": importa_costes}
    else:
        if importa_costes:
            scoring_cv = "f1_weighted"
            plan = {"modo": "multiclase_costes", "desbalance": desbalance}
        else:
            scoring_cv = "f1_macro" if desbalance else "accuracy"
            plan = {"modo": "multiclase_desbalance" if desbalance else "multiclase_equilibrio",
                    "desbalance": desbalance}
    print(f">>> Métrica de CV seleccionada: {scoring_cv}")
    return scoring_cv, plan

def plot_confusion(cm, clases, outpath, title="Matriz de confusión (test)"):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Real")
    ax.set_xticks(range(len(clases)))
    ax.set_yticks(range(len(clases)))
    ax.set_xticklabels(clases, rotation=45, ha="right")
    ax.set_yticklabels(clases)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, int(cm[i, j]), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)

def plot_tree_png(modelo, feature_names, class_names, outpath, max_depth_visual=3):
    fig, ax = plt.subplots(figsize=(12, 8))
    plot_tree(
        modelo,
        feature_names=feature_names,
        class_names=class_names,
        filled=True,
        max_depth=max_depth_visual
    )
    ax.set_title("Árbol (vista truncada para legibilidad)")
    fig.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)

def guardar_expected_columns(columns, path: Path):
    out = {"columns": list(columns), "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

def agrupar_importancias_por_prefijo(df_import, sep="___"):
    # df_import: DataFrame con columnas ['feature','importance']
    grupos = {}
    for _, row in df_import.iterrows():
        feat = str(row["feature"])
        imp  = float(row["importance"])
        pref = feat.split(sep)[0] if sep in feat else feat
        grupos[pref] = grupos.get(pref, 0.0) + imp
    out = pd.DataFrame({"prefijo": list(grupos.keys()), "importance_sum": list(grupos.values())})
    return out.sort_values("importance_sum", ascending=False)


In [ ]:
# === DIAGNÓSTICO + DECISIÓN DE MÉTRICAS ===
diag = diagnostico_balance_multiclase(y_train, rare_threshold=CONFIG["rare_threshold"])
classes = np.unique(y_train); K = len(classes)
scoring_cv, plan = decidir_metricas(K, diag, CONFIG)


In [ ]:
# === MODELO Y GRIDSEARCH ===
base = DecisionTreeClassifier(random_state=CONFIG["random_state"])
param_grid = CONFIG["grid"]

cv = StratifiedKFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=CONFIG["random_state"])
grid = GridSearchCV(
    estimator=base,
    param_grid=param_grid,
    scoring=scoring_cv,
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=0
)

grid.fit(X_train, y_train)

print("\n=== Mejor configuración (CV) ===")
print(grid.best_params_)
print(f"Mejor {scoring_cv}: {grid.best_score_:.4f}")

best = grid.best_estimator_
classes_ = list(best.classes_)


In [ ]:
# === PROBABILIDADES / SCORES ===
probs_train = best.predict_proba(X_train) if hasattr(best, "predict_proba") else None
probs_test  = best.predict_proba(X_test)  if hasattr(best, "predict_proba") else None

Train_out = Train.copy()
Test_out  = Test.copy()

if K == 2 and probs_train is not None:
    # índice de la clase positiva: por convención 1 si existe, si no la mayor etiqueta
    try:
        idx_pos = np.where(classes_ == ["1"])[0][0]
    except:
        # fallback: mayor etiqueta (orden lexicográfico de strings)
        idx_pos = np.argmax(classes_)
    Train_out["scores"] = probs_train[:, idx_pos]
    Test_out["scores"]  = probs_test[:, idx_pos]
elif probs_train is not None:
    for i, c in enumerate(classes_):
        Train_out[f"score_{c}"] = probs_train[:, i]
        Test_out[f"score_{c}"]  = probs_test[:, i]

# Guardar scores
Train_out.to_csv(OUTDIR / "T_train_final_objetivo_scores.csv", index=False)
Test_out.to_csv(OUTDIR / "T_test_final_objetivo_scores.csv", index=False)
print("Scores guardados en carpeta de artefactos.")


In [ ]:
# === EVALUACIÓN ===
if K == 2:
    if probs_test is not None:
        # decisión de umbral si desbalance/importa_costes
        desbalance = (plan.get("desbalance", False) or plan.get("importa_costes", False))
        try:
            idx_pos = np.where(classes_ == ["1"])[0][0]
        except:
            idx_pos = np.argmax(classes_)
        p_test = probs_test[:, idx_pos]

        def evaluate_thresholds(y_true, probs, thresholds=np.arange(0.0,1.0,0.01), criterio="f1"):
            rows=[]
            y_true_bin = (pd.Series(y_true).astype(str) == classes_[idx_pos]).astype(int).to_numpy()
            for t in thresholds:
                y_pred = (probs >= t).astype(int)
                acc = accuracy_score(y_true_bin, y_pred)
                prec, rec, f1, _ = precision_recall_fscore_support(y_true_bin, y_pred, average='binary', zero_division=0)
                rows.append({"t":t,"acc":acc,"prec":prec,"rec":rec,"f1":f1})
            df = pd.DataFrame(rows)
            key = "f1" if criterio=="f1" else "acc"
            t_opt = float(df.loc[df[key].idxmax(), "t"])
            return t_opt, df

        criterio = "f1" if desbalance else "acc"
        alpha_opt, thr_df = evaluate_thresholds(y_test, p_test, criterio=criterio)
        y_pred_bin = (p_test >= alpha_opt).astype(int)
        y_true_bin = (pd.Series(y_test).astype(str) == classes_[idx_pos]).astype(int).to_numpy()

        acc = accuracy_score(y_true_bin, y_pred_bin)
        prec, rec, f1, _ = precision_recall_fscore_support(y_true_bin, y_pred_bin, average='binary', zero_division=0)

        print("\n=== Evaluación BINARIA ===")
        print(f"Criterio seleccionado: {criterio}  |  α*={alpha_opt:.3f}")
        print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
        print("Matriz de confusión:\n", confusion_matrix(y_true_bin, y_pred_bin, labels=[0,1]))
    else:
        # sin probabilidades, usar predict directo
        y_pred = best.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='binary', zero_division=0)
        print("\n=== Evaluación BINARIA (sin proba) ===")
        print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
        print("Matriz de confusión:\n", confusion_matrix(y_test, y_pred))
else:
    # Multiclase
    y_pred = best.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    print("\n=== Evaluación MULTICLASE ===")
    if plan["modo"] == "multiclase_equilibrio":
        print(f"[Balance + igual importancia] → Prioriza Accuracy (control: F1_macro).")
        print(f"Accuracy: {acc:.4f} | F1_macro: {f1_m:.4f}")
    elif plan["modo"] == "multiclase_desbalance":
        print(f"[Desbalance + igual importancia] → Prioriza F1_macro (control: Balanced Accuracy).")
        print(f"F1_macro: {f1_m:.4f} | BalancedAccuracy: {bal_acc:.4f} | Accuracy: {acc:.4f}")
    else:  # costes distintos
        print(f"[Importa distinguir clases] → Prioriza F1_weighted + métricas por clase.")
        print(f"F1_weighted: {f1_w:.4f} | Accuracy: {acc:.4f} | F1_macro: {f1_m:.4f}")

    print("\nMatriz de confusión (filas=verdad, cols=pred):")
    cm = confusion_matrix(y_test, y_pred, labels=classes_)
    print(pd.DataFrame(cm, index=[f"true_{c}" for c in classes_], columns=[f"pred_{c}" for c in classes_]))

    # Guardar figura
    plot_confusion(cm, classes_, OUTDIR / "matriz_confusion.png")

    print("\nReporte por clase:")
    report_txt = classification_report(y_test, y_pred, zero_division=0)
    print(report_txt)


In [ ]:
# === IMPORTANCIAS + ÁRBOL (truncado) ===
importancias = getattr(best, "feature_importances_", None)
if importancias is not None:
    imp = pd.DataFrame({"feature": X_train.columns, "importance": importancias}).sort_values("importance", ascending=False)
    imp.to_csv(OUTDIR / "feature_importances.csv", index=False)
    print("feature_importances.csv guardado.")

    # Agregado por prefijo (para dummies) — opcional
    try:
        agg = agrupar_importancias_por_prefijo(imp, sep=CONFIG["SEP"])
        agg.to_csv(OUTDIR / "feature_importances_por_prefijo.csv", index=False)
        print("feature_importances_por_prefijo.csv guardado.")
    except Exception as e:
        print("No se pudo agregar por prefijo:", e)

# Árbol (vista truncada) para lectura humana
try:
    plot_tree_png(best, list(X_train.columns), classes_, OUTDIR / "arbol_truncado.png", max_depth_visual=CONFIG["max_depth_visual"])
    print("arbol_truncado.png guardado.")
except Exception as e:
    print("No se pudo graficar el árbol:", e)


In [ ]:
# === GUARDAR MODELO, COLUMNAS ESPERADAS, RESUMEN ===
import joblib

joblib.dump(best, OUTDIR / "modelo_arbol.pkl")
print("Modelo guardado:", OUTDIR / "modelo_arbol.pkl")

# expected_columns (exacto como en train)
with open(OUTDIR / "expected_columns.json", "w", encoding="utf-8") as f:
    json.dump({"columns": list(X_train.columns), "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")}, f, ensure_ascii=False, indent=2)

# Resumen
resumen = {
    "best_params": grid.best_params_,
    "cv_best_score": grid.best_score_,
    "scoring_cv": grid.scoring,
    "classes": classes_
}
with open(OUTDIR / "resumen_metricas.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)

# classification_report
try:
    with open(OUTDIR / "classification_report.txt", "w", encoding="utf-8") as f:
        if 'report_txt' in locals():
            f.write(report_txt)
        else:
            f.write("Reporte no disponible (binario con umbral y/o sin probas).")
except Exception as e:
    print("No se pudo escribir classification_report.txt:", e)

print("Artefactos clave escritos en:", OUTDIR.resolve())


In [ ]:
# === ZIP DE ARTEFACTOS ===
dst_dir = OUTDIR
zip_path = OUTDIR / "dt_multiclase_artifacts_bundle.zip"

candidates = [
    OUTDIR / "modelo_arbol.pkl",
    OUTDIR / "expected_columns.json",
    OUTDIR / "resumen_metricas.json",
    OUTDIR / "classification_report.txt",
    OUTDIR / "feature_importances.csv",
    OUTDIR / "feature_importances_por_prefijo.csv",
    OUTDIR / "matriz_confusion.png",
    OUTDIR / "arbol_truncado.png",
    OUTDIR / "T_train_final_objetivo_scores.csv",
    OUTDIR / "T_test_final_objetivo_scores.csv",
]

present = [str(f) for f in candidates if f.exists()]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in present:
        zf.write(f, arcname=os.path.basename(f))

print("ZIP creado en:", zip_path)
print("Incluidos:", [os.path.basename(f) for f in present])
